In [2]:
!pip install -U langchain langchain-community langchain-text-splitters faiss-cpu sentence-transformers transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28
  Attempting uninsta

In [5]:
import os

# =============================
# 📦 IMPORTS
# =============================
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.llms import HuggingFacePipeline

from langgraph.graph import StateGraph

from transformers import pipeline

# =============================
# 📂 FAKE SUPPORT DATA
# =============================
data = [
    "Q: How to reset the device? A: Hold the power button for 10 seconds.",
    "Q: Battery not charging A: Check cable and adapter.",
    "Q: App not syncing A: Restart Bluetooth and reopen app.",
    "Q: Warranty policy A: 1-year replacement warranty.",
    "Q: Device overheating A: Turn off device and let it cool.",
    "Q: Screen not working A: Restart device or update firmware.",
    "Q: Cannot login A: Reset password using email.",
    "Q: Data not updating A: Check internet connection.",
    "Q: Device not turning on A: Charge for 30 minutes.",
    "Q: Strap broken A: Contact support for replacement."
]

docs = [Document(page_content=d) for d in data]

# =============================
# ✂️ CHUNKING
# =============================
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = splitter.split_documents(docs)

# =============================
# 🧠 LOCAL EMBEDDINGS
# =============================
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = FAISS.from_documents(chunks, embeddings)
retriever = db.as_retriever()

# =============================
# 🤖 LOCAL LLM
# =============================
pipe = pipeline("text-generation", model="google/flan-t5-base", max_length=200)
llm = HuggingFacePipeline(pipeline=pipe)

# =============================
# ⚙️ GRAPH FUNCTIONS
# =============================
def retrieve(state):
    query = state["query"]
    docs = retriever.invoke(query)
    state["context"] = " ".join([d.page_content for d in docs])
    return state

def generate(state):
    prompt = f"""
Answer the question using the context below.
If unsure, say "I don't know".

Context:
{state['context']}

Question:
{state['query']}
"""
    response = llm.invoke(prompt)
    state["answer"] = response
    return state

def check_confidence(state):
    if "I don't know" in state["answer"]:
        state["confidence"] = "low"
    else:
        state["confidence"] = "high"
    return state

def human_review(state):
    print("\n⚠️ LOW CONFIDENCE ANSWER ⚠️")
    print("AI Answer:", state["answer"])

    choice = input("Approve? (y/n): ")

    if choice == "y":
        return state
    else:
        new_answer = input("Enter corrected answer: ")
        state["answer"] = new_answer
        return state

# =============================
# 🔁 LANGGRAPH WORKFLOW
# =============================
workflow = StateGraph(dict)

workflow.add_node("retrieve", retrieve)
workflow.add_node("generate", generate)
workflow.add_node("check", check_confidence)
workflow.add_node("human", human_review)

workflow.set_entry_point("retrieve")

workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", "check")

def route(state):
    if state["confidence"] == "low":
        return "human"
    else:
        return "__end__"

workflow.add_conditional_edges("check", route)
workflow.add_edge("human", "__end__")

app = workflow.compile()

# =============================
# ▶️ RUN LOOP
# =============================
while True:
    query = input("\nAsk a question (or type 'exit'): ")

    if query.lower() == "exit":
        break

    result = app.invoke({"query": query})

    print("\n✅ FINAL ANSWER:")
    print(result["answer"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM',


Ask a question (or type 'exit'): how to reset device?

⚠️ LOW CONFIDENCE ANSWER ⚠️
AI Answer: 
Answer the question using the context below.
If unsure, say "I don't know".

Context:
Q: How to reset the device? A: Hold the power button for 10 seconds. Q: Device overheating A: Turn off device and let it cool. Q: Screen not working A: Restart device or update firmware. Q: Device not turning on A: Charge for 30 minutes.

Question:
how to reset device?

Approve? (y/n): no
Enter corrected answer: i din't know

✅ FINAL ANSWER:
i din't know

Ask a question (or type 'exit'): exit
